# Are there more road accidents on Friday the 13th? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/friday-the-13th/story_002_friday_13th.ipynb)

This R notebook recomputes every statistic in
[Are there more road accidents on Friday the 13th?](https://pointzerofive.net/2026/09/friday-the-13th),
story #002 on *pointzerofive*, from the daily collision counts of Great Britain (STATS19).
Colab opens it with the R runtime; run it top to bottom (`Runtime → Run all`).

In [ ]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/friday-the-13th/"
REPO <- ".."
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}
daily <- read.csv(fetch_path("notebooks/raw/stats19/daily_collisions.csv"), stringsAsFactors = FALSE)
daily$date <- as.Date(daily$date)
hol <- as.Date(read.csv(fetch_path("notebooks/raw/stats19/bank_holidays_eng.csv"), stringsAsFactors = FALSE)$date)
published <- fromJSON(fetch_path("data/friday-the-13th.json"), simplifyVector = FALSE)
c(days = nrow(daily), collisions = sum(daily$all), bank_holidays = length(hol))

## 1. Fridays the 13th and their comparison days

For every month with a Friday the 13th: the 13th and its comparison days in the same month
(all other days of the month, the Fridays a week before and after, or only the 6th). Bank holidays
are left out or kept.

In [ ]:
wday <- function(x) as.integer(format(x, "%u"))
fridays_13th <- function(first, last) {
  days <- as.Date(sprintf("%d-%02d-13", rep(first:last, each = 12), rep(1:12, last - first + 1)))
  days[wday(days) == 5]
}
comparison_days <- function(d13, comparison) {
  if (comparison == "all_days") {
    m <- seq(as.Date(format(d13, "%Y-%m-01")), by = "day", length.out = 31)
    m <- m[format(m, "%m") == format(d13, "%m")]
    return(m[m != d13])
  }
  if (comparison == "fridays") return(d13 + c(-7, 7))
  d13 - 7
}
path_data <- function(outcome, comparison, hol) {
  count <- setNames(daily[[outcome]], as.character(daily$date))
  d13s <- fridays_13th(as.integer(format(min(daily$date), "%Y")), as.integer(format(max(daily$date), "%Y")))
  rows <- list()
  for (d13 in as.list(d13s)) {
    if (!is.null(hol) && d13 %in% hol) next
    days <- comparison_days(d13, comparison)
    if (!is.null(hol)) days <- days[!days %in% hol]
    if (!length(days)) next
    rows[[length(rows) + 1]] <- data.frame(month = format(d13), is13 = c(1, rep(0, length(days))),
                                           count = unname(count[as.character(c(d13, days))]))
  }
  do.call(rbind, rows)
}
main_data <- path_data("all", "fridays", hol)
c(fridays13 = sum(main_data$is13), on13 = sum(main_data$count[main_data$is13 == 1]),
  comparison_fridays = sum(main_data$is13 == 0))

## 2. The test: Poisson regression with overdispersion

Daily collision counts are counts, so the model is a Poisson regression of the count on a dummy
for the 13th and one constant per month (the month constants compare each 13th only with its own
comparison days). Counts of accidents vary more than a Poisson distribution allows, so the
dispersion is estimated (quasi-Poisson) and the test of the dummy is a t-test. The effect in per
cent is 100 · (exp(β) − 1); the 95% interval transforms the interval of β the same way.

In [ ]:
ALPHA <- 0.05
test_path <- function(dat) {
  fit <- glm(count ~ is13 + factor(month), family = quasipoisson, data = dat)
  co <- summary(fit)$coefficients
  b <- co["is13", "Estimate"]; se <- co["is13", "Std. Error"]; df <- fit$df.residual
  q <- qt(1 - ALPHA / 2, df)
  pct <- function(v) 100 * (exp(v) - 1)
  list(effect = pct(b), ci_low = pct(b - q * se), ci_high = pct(b + q * se),
       p = co["is13", "Pr(>|t|)"], n = sum(dat$is13), dispersion = summary(fit)$dispersion)
}
m <- test_path(main_data)
cat(sprintf("Friday the 13th: %+.2f%% collisions, 95%% CI [%+.2f, %+.2f], p = %.3f, dispersion %.2f\n",
            m$effect, m$ci_low, m$ci_high, m$p, m$dispersion))

## 3. All 18 calculation paths

In [ ]:
rows <- list()
for (comparison in c("all_days", "fridays", "friday_6th")) for (outcome in c("all", "ksi", "fatal"))
  for (holidays in c("exclude", "keep")) {
    s <- test_path(path_data(outcome, comparison, if (holidays == "exclude") hol else NULL))
    rows[[length(rows) + 1]] <- data.frame(comparison, outcome, holidays, n = s$n, effect = s$effect,
                                           ci_low = s$ci_low, ci_high = s$ci_high, p = s$p)
  }
specs <- do.call(rbind, rows)
for (i in seq_len(nrow(specs))) {
  pub <- published$specs[[i]]
  stopifnot(identical(unname(unlist(pub$choices)), unname(unlist(specs[i, 1:3]))), pub$n == specs$n[i])
  for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(pub[[k]] - specs[[k]][i]) < 1e-9)
}
print(specs, digits = 3)

## 4. The verdict

Main calculation: Fridays a week before and after, all collisions, bank holidays left out. "Yes"
needs p < 0.05 and more collisions on the 13th.

In [ ]:
verdict <- if (m$p < 0.05 && m$effect > 0) "yes" else "no"
stopifnot(abs(m$effect - published$specs[[which(vapply(published$specs, function(s)
  identical(unlist(s$choices), c(comparison = "fridays", outcome = "all", holidays = "exclude")), TRUE))]]$effect) < 1e-9)
cat("verdict:", verdict, "\n")
cat(sprintf("yes on %d of %d paths\n", sum(specs$p < 0.05 & specs$effect > 0), nrow(specs)))

## Data and methods

- **Source:** Department for Transport, Road Safety Data (STATS19), Open Government Licence v3.0;
  daily counts by `friday13_prep.py`, bank holidays by `friday13_holidays.py`.
- **Pipeline:** `friday13_pipeline.R` (base R and `jsonlite`), console output in
  `friday-the-13th.output.txt`.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.

In [ ]:
cat("All statistics reproduced ✓\n")